# Active-learning results: strategy and methodology audit

This notebook answers three questions:

1. Do coarse/fine uncertainty measure (UM) and entropy measure (EM) outperform random acquisition?
2. What are the strategies actually selecting?
3. Which experiments should run next, and which methodology issues must be fixed first?

The analysis discovers both the intended runs directory and the legacy run directories currently beside this notebook. It inventories incomplete runs instead of silently dropping them.

**Round alignment:** files in round n describe the query made *after* training round n; that batch can only affect the validation result in round n + 1.

**Inference boundary:** all efficacy comparisons here are descriptive validation results. With one seed and nondeterministic training, rounds and clips are not independent replicates and cannot support confidence intervals or significance tests. Event F1 is additionally provisional because Section 8 demonstrates a counting bug in the current evaluator.

## 1. Load a consistent snapshot

In [ ]:
import hashlib
import json
import math
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)

STRATEGY_LABELS = {
    "RANDOM_SAMPLING": "Random",
    "COARSE_UNCERTAINTY_MEASURE": "Coarse UM",
    "COARSE_ENTROPY_MEASURE": "Coarse EM",
    "FINE_UNCERTAINTY_MEASURE": "Fine UM",
    "FINE_ENTROPY_MEASURE": "Fine EM",
}
STRATEGY_ORDER = list(STRATEGY_LABELS.values())
STRATEGY_RANK = {label: rank for rank, label in enumerate(STRATEGY_ORDER)}
STRATEGY_COLORS = {
    "Random": "#7f7f7f",
    "Coarse UM": "#4c78a8",
    "Coarse EM": "#59a14f",
    "Fine UM": "#e45756",
    "Fine EM": "#b279a2",
}
SCORE_KEYS = {
    "COARSE_UNCERTAINTY_MEASURE": "coarse_uncertainty_measure",
    "COARSE_ENTROPY_MEASURE": "coarse_entropy_measure",
    "FINE_UNCERTAINTY_MEASURE": "fine_uncertainty_measure",
    "FINE_ENTROPY_MEASURE": "fine_entropy_measure",
}

cwd = Path.cwd().resolve()
relative_experiment_dir = Path("experiments") / "3_uncertainty_and_entropy"
if cwd.name == "3_uncertainty_and_entropy":
    experiment_dir = cwd
elif (cwd / relative_experiment_dir).is_dir():
    experiment_dir = cwd / relative_experiment_dir
else:
    raise FileNotFoundError(
        "Run this notebook from its own directory or from the repository root."
    )

repo_root = experiment_dir.parents[1]
runs_dir = experiment_dir / "runs"


def read_json(path):
    with path.open() as file:
        return json.load(file)


def parse_round(path):
    return int(path.parent.name.rsplit("_", 1)[1])


# Include config-only runs so an interrupted experiment remains visible.
config_paths = sorted(
    set(runs_dir.glob("*/config.json")) | set(experiment_dir.glob("*/config.json"))
)
if not config_paths:
    raise FileNotFoundError(
        f"No run configs found below {runs_dir} or {experiment_dir}"
    )

run_infos = []
history_records = []
load_errors = []
configs_by_run = {}
histories_by_run = {}
run_dirs_by_run = {}

for config_path in config_paths:
    run_dir = config_path.parent
    run_name = run_dir.name
    try:
        config = read_json(config_path)
    except (OSError, ValueError, TypeError, json.JSONDecodeError) as error:
        load_errors.append((run_name, "config", str(error)))
        continue

    strategy_key = config.get("query_strategy", "UNKNOWN")
    strategy = STRATEGY_LABELS.get(strategy_key, strategy_key.title())
    history_path = run_dir / "history.json"
    history = []
    if history_path.exists():
        try:
            history = read_json(history_path)
            if not isinstance(history, list):
                raise TypeError("history must be a list")
        except (OSError, ValueError, TypeError, json.JSONDecodeError) as error:
            load_errors.append((run_name, "history", str(error)))
            history = []

    round_dirs = sorted(path for path in run_dir.glob("round_*") if path.is_dir())
    active_round = (
        int(round_dirs[-1].name.rsplit("_", 1)[1]) if round_dirs else None
    )
    progress = None
    if round_dirs:
        loss_path = round_dirs[-1] / "loss.json"
        if loss_path.exists():
            try:
                progress = len(read_json(loss_path))
            except (OSError, ValueError, TypeError, json.JSONDecodeError):
                progress = "writing"

    expected_rounds = (
        math.ceil(
            (
                float(config["max_annotation_budget"])
                - float(config["initial_labeled_pool_size"])
            )
            / float(config["query_batch_size"])
        )
        + 1
    )
    is_complete = bool(history) and (
        int(history[-1]["labeled_pool_frames"])
        >= int(config["max_annotation_frame_budget"])
    )
    if is_complete:
        status = "complete"
    elif active_round is None:
        status = "configured; no round artifacts"
    elif progress == "writing":
        status = f"round {active_round} is being written"
    elif progress is not None and (
        not history or active_round > int(history[-1]["round"])
    ):
        status = (
            f"training round {active_round} "
            f"({progress}/{config['f3ed']['num_epochs']} epochs saved)"
        )
    else:
        status = f"partial ({len(history)}/{expected_rounds} evaluation rounds)"

    configs_by_run[run_name] = config
    histories_by_run[run_name] = history
    run_dirs_by_run[run_name] = run_dir
    run_infos.append(
        {
            "run": run_name,
            "strategy": strategy,
            "strategy_key": strategy_key,
            "seed": int(config["seed"]),
            "pooling": str(config.get("query_score_pooling", "NA")).lower(),
            "status": status,
            "evaluation_rounds": len(history),
            "expected_rounds": expected_rounds,
            "last_budget_pct": (
                float(history[-1]["labeled_budget_percent"]) if history else np.nan
            ),
            "active_round": active_round,
            "saved_epochs_in_active_round": progress,
            "run_dir": run_dir,
        }
    )

    for row in history:
        history_records.append(
            {
                "run": run_name,
                "strategy": strategy,
                "strategy_key": strategy_key,
                "seed": int(config["seed"]),
                "pooling": str(config.get("query_score_pooling", "NA")).lower(),
                "round": int(row["round"]),
                "labeled_budget_pct": float(row["labeled_budget_percent"]),
                "labeled_frames": int(row["labeled_pool_frames"]),
                "labeled_clips": int(row["labeled_pool_size"]),
                "best_epoch": int(row["best_epoch"]),
                "val_edit": float(row["val_edit"]),
                "val_f1_event": float(row["val_f1_event"]),
                "val_f1_element": float(row["val_f1_element"]),
            }
        )

inventory_df = pd.DataFrame(run_infos)
results_df = pd.DataFrame(history_records)
if results_df.empty:
    raise FileNotFoundError("No completed evaluation round was found.")

inventory_df["strategy_rank"] = inventory_df["strategy"].map(STRATEGY_RANK)
inventory_df = inventory_df.sort_values(["strategy_rank", "seed", "run"]).drop(
    columns="strategy_rank"
)
results_df["strategy_rank"] = results_df["strategy"].map(STRATEGY_RANK)
results_df = results_df.sort_values(
    ["strategy_rank", "seed", "round", "run"]
).drop(columns="strategy_rank")

print(f"Experiment directory: {experiment_dir}")
print(
    f"Snapshot loaded {len(inventory_df)} configured runs, "
    f"{results_df['run'].nunique()} with history, and {len(results_df)} evaluations."
)
if not any(runs_dir.glob("*")):
    print("Note: runs/ is empty; results were discovered in legacy sibling directories.")
if load_errors:
    print("Files skipped because the snapshot caught them mid-write or malformed:")
    for run, artifact, reason in load_errors:
        print(f"  {run} / {artifact}: {reason}")

## 2. Run completeness and comparability

A valid strategy comparison needs the same protocol and starting pool. It also needs training noise to be small relative to acquisition effects. The tables below check all three.

In [ ]:
inventory_view = inventory_df[
    [
        "strategy",
        "seed",
        "pooling",
        "status",
        "evaluation_rounds",
        "expected_rounds",
        "last_budget_pct",
    ]
].rename(
    columns={
        "strategy": "Strategy",
        "seed": "Seed",
        "pooling": "Pooling",
        "status": "Status",
        "evaluation_rounds": "Evaluated rounds",
        "expected_rounds": "Expected rounds",
        "last_budget_pct": "Last budget (%)",
    }
)
display(inventory_view.style.format({"Last budget (%)": "{:.3f}"}))

protocol_fields = [
    "initial_labeled_pool_size",
    "query_batch_size",
    "max_annotation_budget",
    "active_learning_budget_unit",
    "total_training_frames",
    "query_score_pooling",
]
protocol_rows = []
for run_name, config in configs_by_run.items():
    protocol_rows.append(
        {
            "run": run_name,
            **{field: config.get(field) for field in protocol_fields},
        }
    )
protocol_df = pd.DataFrame(protocol_rows)
protocol_variation = pd.DataFrame(
    {
        "Distinct values": protocol_df[protocol_fields].nunique(dropna=False),
        "Values": [
            ", ".join(map(str, protocol_df[field].drop_duplicates().tolist()))
            for field in protocol_fields
        ],
    }
)
display(Markdown("**Protocol fields across runs**"))
display(protocol_variation)

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


round0_rows = []
for info in run_infos:
    run_name = info["run"]
    labeled_path = info["run_dir"] / "round_000" / "labeled_train.json"
    round0_result = results_df[
        (results_df["run"] == run_name) & (results_df["round"] == 0)
    ]
    row = round0_result.iloc[0] if not round0_result.empty else None
    round0_rows.append(
        {
            "Strategy": info["strategy"],
            "Seed": info["seed"],
            "Labeled-pool SHA256": (
                sha256(labeled_path)[:12] if labeled_path.exists() else "missing"
            ),
            "Edit": row["val_edit"] if row is not None else np.nan,
            "Event F1": row["val_f1_event"] if row is not None else np.nan,
            "Element F1": row["val_f1_element"] if row is not None else np.nan,
        }
    )

round0_df = pd.DataFrame(round0_rows)
round0_df["_rank"] = round0_df["Strategy"].map(STRATEGY_RANK)
round0_df = round0_df.sort_values(["_rank", "Seed"]).drop(columns="_rank")
display(Markdown("**Same-data round-0 reproducibility check**"))
display(
    round0_df.style.format(
        {"Edit": "{:.3f}", "Event F1": "{:.4f}", "Element F1": "{:.4f}"},
        na_rep="not evaluated yet",
    )
)

evaluated_round0 = round0_df.dropna(subset=["Edit"])
metric_ranges = {
    metric: evaluated_round0[metric].max() - evaluated_round0[metric].min()
    for metric in ["Edit", "Event F1", "Element F1"]
}
unique_pool_hashes = round0_df.loc[
    round0_df["Labeled-pool SHA256"] != "missing", "Labeled-pool SHA256"
].nunique()
seed_count = inventory_df["seed"].nunique()
max_budget_spread = (
    results_df.groupby("round")["labeled_budget_pct"]
    .agg(lambda values: values.max() - values.min())
    .max()
)

transition_rows = []
for info in run_infos:
    run_dir = info["run_dir"]
    history_lookup = {
        int(row["round"]): row for row in histories_by_run[info["run"]]
    }
    for query_path in sorted(run_dir.glob("round_*/queried_samples.json")):
        query_round = parse_round(query_path)
        next_labeled_path = (
            run_dir / f"round_{query_round + 1:03d}" / "labeled_train.json"
        )
        current_labeled_path = query_path.parent / "labeled_train.json"
        if not next_labeled_path.exists() or query_round + 1 not in history_lookup:
            continue
        current_annotations = read_json(current_labeled_path)
        next_annotations = read_json(next_labeled_path)
        queried_videos = set(read_json(query_path))
        current_videos = {row["video"] for row in current_annotations}
        next_videos = {row["video"] for row in next_annotations}
        expected_next = current_videos | queried_videos
        expected_frames = sum(int(row["num_frames"]) for row in next_annotations)
        recorded_frames = int(history_lookup[query_round + 1]["labeled_pool_frames"])
        transition_rows.append(
            {
                "Strategy": info["strategy"],
                "Seed": info["seed"],
                "Query round": query_round,
                "No reselection": current_videos.isdisjoint(queried_videos),
                "Exact next pool": next_videos == expected_next,
                "Frame count matches history": expected_frames == recorded_frames,
            }
        )

transition_df = pd.DataFrame(transition_rows)
if not transition_df.empty:
    transition_summary = (
        transition_df.groupby(["Strategy", "Seed"], sort=False)
        .agg(
            Transitions=("Query round", "size"),
            No_reselection=("No reselection", "all"),
            Exact_pool_update=("Exact next pool", "all"),
            Frame_count_consistent=("Frame count matches history", "all"),
        )
        .reset_index()
    )
    transition_summary["_rank"] = transition_summary["Strategy"].map(STRATEGY_RANK)
    transition_summary = transition_summary.sort_values(["_rank", "Seed"]).drop(
        columns="_rank"
    )
    display(Markdown("**Round-transition integrity**"))
    display(transition_summary)

display(
    Markdown(
        f"""
**What this establishes**

- Unique round-0 labeled-pool hashes: **{unique_pool_hashes}** (one means every strategy starts from identical annotations).
- Independent experimental seeds: **{seed_count}**.
- Despite the shared pool, round-0 score ranges are **{metric_ranges['Edit']:.3f} edit**, **{100 * metric_ranges['Event F1']:.2f} event-F1 points**, and **{100 * metric_ranges['Element F1']:.2f} element-F1 points**.
- Whole-clip selection changes nominal budgets only slightly: the largest between-strategy spread at a round is **{max_budget_spread:.3f} percentage points**.
- Passing transition checks show that saved queries are added exactly once and recorded frame totals are internally consistent. They do not establish that a query rule improves generalization.
"""
    )
)

## 3. Validation learning curves

The x-axis uses the realized percentage of labeled training frames, not the nominal round number. A point at round 0 is pre-acquisition; the first strategy-dependent evaluation is round 1.

In [ ]:
METRICS = {
    "val_edit": ("Validation edit score", "Edit score"),
    "val_f1_event": ("Validation event F1", "F1"),
    "val_f1_element": ("Validation element F1", "F1"),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
for ax, (metric, (title, ylabel)) in zip(axes, METRICS.items()):
    for strategy in STRATEGY_ORDER:
        strategy_data = results_df[results_df["strategy"] == strategy]
        if strategy_data.empty:
            continue

        for _, run_data in strategy_data.groupby("run"):
            ax.plot(
                run_data["labeled_budget_pct"],
                run_data[metric],
                color=STRATEGY_COLORS[strategy],
                alpha=0.20,
                linewidth=1,
            )

        summary = (
            strategy_data.groupby("round")
            .agg(
                budget=("labeled_budget_pct", "mean"),
                mean=(metric, "mean"),
                std=(metric, "std"),
                n=("run", "nunique"),
            )
            .reset_index()
        )
        ax.plot(
            summary["budget"],
            summary["mean"],
            marker="o",
            markersize=4.5,
            linewidth=2.2,
            color=STRATEGY_COLORS[strategy],
            label=strategy,
        )
        if summary["std"].notna().any():
            lower = summary["mean"] - summary["std"].fillna(0)
            upper = summary["mean"] + summary["std"].fillna(0)
            ax.fill_between(
                summary["budget"],
                lower,
                upper,
                color=STRATEGY_COLORS[strategy],
                alpha=0.12,
            )

    ax.set(
        title=title,
        xlabel="Labeled training frames (% of full train set)",
        ylabel=ylabel,
    )
    ax.grid(alpha=0.25)

axes[0].legend(title="Acquisition strategy", fontsize=8)
fig.suptitle(
    "Validation performance versus realized annotation budget",
    y=1.02,
    fontsize=14,
)
fig.tight_layout()
plt.show()

## 4. Effect sizes relative to random

Two summaries are useful:

- **Budget-normalized AULC**: the mean height of each learning curve over its observed budget range.
- **Paired round delta**: active minus random for the same seed and round. F1 differences are shown in percentage points.

Round 0 is excluded from post-query deltas. With only one seed, these are point estimates—not uncertainty estimates.

In [ ]:
def normalized_aulc(run_data, metric):
    ordered = run_data.sort_values("labeled_budget_pct")
    x = ordered["labeled_budget_pct"].to_numpy()
    y = ordered[metric].to_numpy()
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapz(y, x) / (x[-1] - x[0]))


aulc_rows = []
for run_name, run_data in results_df.groupby("run"):
    first = run_data.iloc[0]
    aulc_rows.append(
        {
            "run": run_name,
            "Strategy": first["strategy"],
            "Seed": int(first["seed"]),
            "Rounds": len(run_data),
            "AULC edit": normalized_aulc(run_data, "val_edit"),
            "AULC event F1": normalized_aulc(run_data, "val_f1_event"),
            "AULC element F1": normalized_aulc(run_data, "val_f1_element"),
        }
    )
aulc_df = pd.DataFrame(aulc_rows)
random_aulc = (
    aulc_df[aulc_df["Strategy"] == "Random"]
    .groupby("Seed")[["AULC edit", "AULC event F1", "AULC element F1"]]
    .mean()
    .add_suffix(" random")
)
aulc_comparison = aulc_df.join(random_aulc, on="Seed")
for metric in ["edit", "event F1", "element F1"]:
    aulc_comparison[f"Delta AULC {metric}"] = (
        aulc_comparison[f"AULC {metric}"]
        - aulc_comparison[f"AULC {metric} random"]
    )
aulc_view = aulc_comparison[
    [
        "Strategy",
        "Seed",
        "Rounds",
        "AULC edit",
        "Delta AULC edit",
        "AULC event F1",
        "Delta AULC event F1",
        "AULC element F1",
        "Delta AULC element F1",
    ]
].copy()
aulc_view["_rank"] = aulc_view["Strategy"].map(STRATEGY_RANK)
aulc_view = aulc_view.sort_values(["_rank", "Seed"]).drop(columns="_rank")
display(
    aulc_view.style.format(
        {
            "AULC edit": "{:.3f}",
            "Delta AULC edit": "{:+.3f}",
            "AULC event F1": "{:.4f}",
            "Delta AULC event F1": "{:+.4f}",
            "AULC element F1": "{:.4f}",
            "Delta AULC element F1": "{:+.4f}",
        },
        na_rep="insufficient rounds",
    )
)

random_rounds = (
    results_df[results_df["strategy"] == "Random"]
    .groupby(["seed", "round"], as_index=False)[
        ["val_edit", "val_f1_event", "val_f1_element"]
    ]
    .mean()
)
active_rounds = (
    results_df[results_df["strategy"] != "Random"]
    .groupby(["strategy", "seed", "round"], as_index=False)[
        ["val_edit", "val_f1_event", "val_f1_element"]
    ]
    .mean()
)
paired_df = active_rounds.merge(
    random_rounds,
    on=["seed", "round"],
    suffixes=("", "_random"),
    validate="many_to_one",
)
for metric in ["val_edit", "val_f1_event", "val_f1_element"]:
    paired_df[f"delta_{metric}"] = paired_df[metric] - paired_df[f"{metric}_random"]

post_query_df = paired_df[paired_df["round"] > 0].copy()
delta_rows = []
for strategy, strategy_data in post_query_df.groupby("strategy"):
    latest_by_seed = (
        strategy_data.sort_values("round").groupby("seed", as_index=False).tail(1)
    )
    mean_by_seed = strategy_data.groupby("seed")[
        ["delta_val_edit", "delta_val_f1_event", "delta_val_f1_element"]
    ].mean()
    delta_rows.append(
        {
            "Strategy": strategy,
            "Seeds": strategy_data["seed"].nunique(),
            "Post-query comparisons": len(strategy_data),
            "Latest matched round": ", ".join(
                map(str, sorted(latest_by_seed["round"].unique()))
            ),
            "Final delta edit": latest_by_seed["delta_val_edit"].mean(),
            "Final delta event F1 (pp)": (
                100 * latest_by_seed["delta_val_f1_event"].mean()
            ),
            "Final delta element F1 (pp)": (
                100 * latest_by_seed["delta_val_f1_element"].mean()
            ),
            "Mean post-query delta edit": mean_by_seed["delta_val_edit"].mean(),
            "Mean post-query delta event F1 (pp)": (
                100 * mean_by_seed["delta_val_f1_event"].mean()
            ),
            "Mean post-query delta element F1 (pp)": (
                100 * mean_by_seed["delta_val_f1_element"].mean()
            ),
        }
    )

delta_summary = pd.DataFrame(delta_rows)
if not delta_summary.empty:
    delta_summary["_rank"] = delta_summary["Strategy"].map(STRATEGY_RANK)
    delta_summary = delta_summary.sort_values("_rank").drop(columns="_rank")
    display(
        delta_summary.style.format(
            {
                "Final delta edit": "{:+.3f}",
                "Final delta event F1 (pp)": "{:+.2f}",
                "Final delta element F1 (pp)": "{:+.2f}",
                "Mean post-query delta edit": "{:+.3f}",
                "Mean post-query delta event F1 (pp)": "{:+.2f}",
                "Mean post-query delta element F1 (pp)": "{:+.2f}",
            }
        )
    )

event_delta_by_round = (
    post_query_df.pivot_table(
        index="strategy",
        columns="round",
        values="delta_val_f1_event",
        aggfunc="mean",
    )
    * 100
)
event_delta_by_round = event_delta_by_round.reindex(
    [strategy for strategy in STRATEGY_ORDER if strategy in event_delta_by_round.index]
)
event_delta_by_round.columns = [
    f"Round {round_number}" for round_number in event_delta_by_round.columns
]
display(Markdown("**Event-F1 delta versus random by evaluation round (percentage points)**"))
display(
    event_delta_by_round.style.format("{:+.2f}").background_gradient(
        cmap="RdYlGn", axis=None, vmin=-4, vmax=4
    )
)

In [ ]:
readout_lines = ["**Descriptive efficacy readout**", ""]
for strategy in STRATEGY_ORDER[1:]:
    strategy_data = post_query_df[post_query_df["strategy"] == strategy]
    if strategy_data.empty:
        readout_lines.append(
            f"- **{strategy}: unknown.** No post-acquisition evaluation is available."
        )
        continue
    latest = (
        strategy_data.sort_values("round").groupby("seed", as_index=False).tail(1)
    )
    edit_delta = latest["delta_val_edit"].mean()
    event_delta = 100 * latest["delta_val_f1_event"].mean()
    element_delta = 100 * latest["delta_val_f1_element"].mean()
    if event_delta > 0 and element_delta > 0:
        signal = "positive F1 signal, but not proof of a win"
    elif event_delta < 0 and element_delta < 0:
        signal = "currently trails random on both F1 metrics"
    else:
        signal = "mixed signal"
    readout_lines.append(
        f"- **{strategy}: {signal}.** Latest matched deltas are "
        f"{edit_delta:+.2f} edit, {event_delta:+.2f} event-F1 points, and "
        f"{element_delta:+.2f} element-F1 points."
    )
readout_lines.extend(
    [
        "",
        (
            f"Only **{seed_count} seed** is present, while identical-data round-0 "
            f"edit scores span **{metric_ranges['Edit']:.2f} points**. "
            "The correct overall verdict is therefore **none of the strategies is "
            "yet demonstrated to beat random**."
        ),
    ]
)
display(Markdown("\n".join(readout_lines)))

## 5. Are UM and EM meaningfully different query rules?

For a single binary probability, UM and binary entropy are monotone transforms of the same distance from 0.5. Mean pooling can reorder clips, but very high rank correlation and high same-model top-batch overlap would make a full UM-versus-EM experiment grid low-value.

Fine scores have an additional limitation in the implementation: they are pooled only over frames that the coarse head already predicts as events. A missed event contributes no fine uncertainty, and a clip with no predicted event receives a fine score of zero.

The next cell uses each saved model/pool snapshot to:

- compute rank correlations;
- counterfactually select a frame-budget-matched batch using every saved score column;
- verify that the actual selected flags equal the expected top-score prefix;
- report the share of clips with zero fine scores.

In [ ]:
def rank_correlation(frame, left, right):
    return frame[left].rank(method="average").corr(
        frame[right].rank(method="average")
    )


def select_score_prefix(frame, score_column, frame_budget):
    ordered = frame.sort_values(
        [score_column, "index"],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    cumulative_frames = ordered["num_frames"].cumsum().to_numpy()
    cutoff = int(np.searchsorted(cumulative_frames, frame_budget, side="left")) + 1
    cutoff = min(cutoff, len(ordered))
    return set(ordered.iloc[:cutoff]["video"])


def jaccard(left, right):
    union = left | right
    return len(left & right) / len(union) if union else np.nan


score_diagnostics = []
for info in run_infos:
    run_name = info["run"]
    run_dir = info["run_dir"]
    config = configs_by_run[run_name]
    history_lookup = {
        int(row["round"]): row for row in histories_by_run[run_name]
    }
    for score_path in sorted(run_dir.glob("round_*/query_scores.json")):
        query_round = parse_round(score_path)
        unlabeled_path = score_path.parent / "unlabeled_pool.json"
        if query_round not in history_lookup or not unlabeled_path.exists():
            continue
        try:
            score_frame = pd.DataFrame(read_json(score_path))
            unlabeled = read_json(unlabeled_path)
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            continue

        required_scores = list(SCORE_KEYS.values())
        if score_frame.empty or not set(required_scores).issubset(score_frame.columns):
            continue
        frames_by_video = {
            row["video"]: int(row["num_frames"]) for row in unlabeled
        }
        score_frame["num_frames"] = score_frame["video"].map(frames_by_video)
        if score_frame["num_frames"].isna().any():
            raise ValueError(f"Could not match every scored video in {score_path}")

        current_labeled_frames = int(
            history_lookup[query_round]["labeled_pool_frames"]
        )
        target_frames = min(
            int(config["query_batch_frame_budget"]),
            int(config["max_annotation_frame_budget"]) - current_labeled_frames,
        )
        selected_sets = {
            score_key: select_score_prefix(score_frame, score_key, target_frames)
            for score_key in required_scores
        }
        actual_selected = set(score_frame.loc[score_frame["selected"], "video"])
        actual_key = SCORE_KEYS.get(info["strategy_key"])
        actual_selected_frames = int(
            score_frame.loc[score_frame["selected"], "num_frames"].sum()
        )

        coarse_cross_fine = [
            jaccard(selected_sets[coarse], selected_sets[fine])
            for coarse in [
                "coarse_uncertainty_measure",
                "coarse_entropy_measure",
            ]
            for fine in [
                "fine_uncertainty_measure",
                "fine_entropy_measure",
            ]
        ]
        score_diagnostics.append(
            {
                "Strategy": info["strategy"],
                "Seed": info["seed"],
                "Query round": query_round,
                "Pool clips": len(score_frame),
                "Selected clips": len(actual_selected),
                "Target frames": target_frames,
                "Selected frames": actual_selected_frames,
                "Budget overshoot (%)": (
                    100
                    * (actual_selected_frames - target_frames)
                    / target_frames
                ),
                "Coarse UM/EM rank correlation": rank_correlation(
                    score_frame,
                    "coarse_uncertainty_measure",
                    "coarse_entropy_measure",
                ),
                "Fine UM/EM rank correlation": rank_correlation(
                    score_frame,
                    "fine_uncertainty_measure",
                    "fine_entropy_measure",
                ),
                "Coarse UM/EM top-batch Jaccard": jaccard(
                    selected_sets["coarse_uncertainty_measure"],
                    selected_sets["coarse_entropy_measure"],
                ),
                "Fine UM/EM top-batch Jaccard": jaccard(
                    selected_sets["fine_uncertainty_measure"],
                    selected_sets["fine_entropy_measure"],
                ),
                "Mean coarse/fine top-batch Jaccard": float(
                    np.mean(coarse_cross_fine)
                ),
                "Zero fine-score clips (%)": (
                    100
                    * (
                        (score_frame["fine_uncertainty_measure"] <= 1e-12)
                        & (score_frame["fine_entropy_measure"] <= 1e-12)
                    ).mean()
                ),
                "Actual selection is exact score prefix": (
                    actual_key is not None
                    and actual_selected == selected_sets[actual_key]
                ),
            }
        )

score_diag_df = pd.DataFrame(score_diagnostics)
if score_diag_df.empty:
    display(Markdown("No complete query-score snapshots are available."))
else:
    score_diag_summary = (
        score_diag_df.groupby(["Strategy", "Seed"], sort=False)
        .agg(
            Snapshots=("Query round", "size"),
            Coarse_rank_rho=("Coarse UM/EM rank correlation", "mean"),
            Fine_rank_rho=("Fine UM/EM rank correlation", "mean"),
            Coarse_batch_Jaccard=("Coarse UM/EM top-batch Jaccard", "mean"),
            Fine_batch_Jaccard=("Fine UM/EM top-batch Jaccard", "mean"),
            Coarse_fine_batch_Jaccard=(
                "Mean coarse/fine top-batch Jaccard",
                "mean",
            ),
            Zero_fine_score_clips_pct=("Zero fine-score clips (%)", "mean"),
            Mean_budget_overshoot_pct=("Budget overshoot (%)", "mean"),
            Exact_saved_ranking=("Actual selection is exact score prefix", "all"),
        )
        .reset_index()
    )
    score_diag_summary["_rank"] = score_diag_summary["Strategy"].map(STRATEGY_RANK)
    score_diag_summary = score_diag_summary.sort_values(["_rank", "Seed"]).drop(
        columns="_rank"
    )
    display(
        score_diag_summary.style.format(
            {
                "Coarse_rank_rho": "{:.3f}",
                "Fine_rank_rho": "{:.3f}",
                "Coarse_batch_Jaccard": "{:.3f}",
                "Fine_batch_Jaccard": "{:.3f}",
                "Coarse_fine_batch_Jaccard": "{:.3f}",
                "Zero_fine_score_clips_pct": "{:.1f}",
                "Mean_budget_overshoot_pct": "{:.3f}",
            }
        )
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for strategy in STRATEGY_ORDER:
        plot_data = score_diag_df[score_diag_df["Strategy"] == strategy]
        if plot_data.empty:
            continue
        color = STRATEGY_COLORS[strategy]
        axes[0].plot(
            plot_data["Query round"],
            plot_data["Coarse UM/EM rank correlation"],
            marker="o",
            color=color,
            label=f"{strategy}: coarse",
        )
        axes[0].plot(
            plot_data["Query round"],
            plot_data["Fine UM/EM rank correlation"],
            marker="x",
            linestyle="--",
            color=color,
            label=f"{strategy}: fine",
        )
        axes[1].plot(
            plot_data["Query round"],
            plot_data["Coarse UM/EM top-batch Jaccard"],
            marker="o",
            color=color,
            label=f"{strategy}: coarse",
        )
        axes[1].plot(
            plot_data["Query round"],
            plot_data["Fine UM/EM top-batch Jaccard"],
            marker="x",
            linestyle="--",
            color=color,
            label=f"{strategy}: fine",
        )
        axes[2].plot(
            plot_data["Query round"],
            plot_data["Zero fine-score clips (%)"],
            marker="o",
            color=color,
            label=strategy,
        )

    axes[0].set(
        title="UM/EM rank similarity",
        xlabel="Query round",
        ylabel="Spearman rank correlation",
        ylim=(0, 1.02),
    )
    axes[1].set(
        title="Same-model selected-batch similarity",
        xlabel="Query round",
        ylabel="Jaccard overlap",
        ylim=(0, 1.02),
    )
    axes[2].set(
        title="Fine score gating",
        xlabel="Query round",
        ylabel="Clips with both fine scores zero (%)",
    )
    for ax in axes:
        ax.grid(alpha=0.25)
    axes[2].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

round0_selected = {}
for info in run_infos:
    query_path = info["run_dir"] / "round_000" / "queried_samples.json"
    if query_path.exists():
        try:
            round0_selected[info["strategy"]] = set(read_json(query_path))
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            pass

actual_overlap_rows = []
for left, right in combinations(
    [strategy for strategy in STRATEGY_ORDER if strategy in round0_selected], 2
):
    actual_overlap_rows.append(
        {
            "Strategy A": left,
            "Strategy B": right,
            "Shared clips": len(round0_selected[left] & round0_selected[right]),
            "Jaccard": jaccard(round0_selected[left], round0_selected[right]),
        }
    )
if actual_overlap_rows:
    display(Markdown("**Actual round-0 batch overlap across separately trained runs**"))
    display(
        pd.DataFrame(actual_overlap_rows).style.format({"Jaccard": "{:.3f}"})
    )

## 6. What did each strategy buy?

This is an **oracle/post-hoc diagnostic** because it reads the ground-truth annotations of queried clips. It must never be used while selecting from the unlabeled pool.

Event density helps detect a failure mode in which a fine strategy mostly selects clips that contain few actual events. It is not sufficient by itself: rare labels, redundancy, and annotation difficulty also matter.

In [ ]:
dataset_dir = repo_root / "src" / "F3Set" / "data" / "f3set-tennis"
train_annotations = read_json(dataset_dir / "train.json")
train_by_video = {row["video"]: row for row in train_annotations}
if len(train_by_video) != len(train_annotations):
    raise ValueError("Training video identifiers are not unique.")

acquisition_rows = []
seen_by_run = {}
for info in run_infos:
    seen = set()
    for query_path in sorted(info["run_dir"].glob("round_*/queried_samples.json")):
        query_round = parse_round(query_path)
        try:
            queried_videos = read_json(query_path)
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            continue
        annotations = [train_by_video[video] for video in queried_videos]
        frame_count = sum(int(row["num_frames"]) for row in annotations)
        event_count = sum(len(row.get("events", [])) for row in annotations)
        labels = {
            event["label"]
            for row in annotations
            for event in row.get("events", [])
        }
        queried_set = set(queried_videos)
        acquisition_rows.append(
            {
                "Strategy": info["strategy"],
                "Seed": info["seed"],
                "Query round": query_round,
                "Affects evaluation round": query_round + 1,
                "Clips": len(annotations),
                "Frames": frame_count,
                "Events": event_count,
                "Events per 1k frames": 1000 * event_count / frame_count,
                "Clips with no GT event (%)": (
                    100
                    * sum(not row.get("events") for row in annotations)
                    / len(annotations)
                ),
                "Unique event labels": len(labels),
                "Reselected clips": len(seen & queried_set),
            }
        )
        seen.update(queried_set)
    seen_by_run[info["run"]] = seen

acquisition_df = pd.DataFrame(acquisition_rows)
if acquisition_df.empty:
    display(Markdown("No completed acquisitions are available."))
else:
    acquisition_summary = (
        acquisition_df.groupby(["Strategy", "Seed"], sort=False)
        .agg(
            Query_batches=("Query round", "size"),
            Mean_clips_per_batch=("Clips", "mean"),
            Mean_events_per_1k_frames=("Events per 1k frames", "mean"),
            Mean_eventless_clips_pct=("Clips with no GT event (%)", "mean"),
            Mean_unique_labels_per_batch=("Unique event labels", "mean"),
            Reselected_clips=("Reselected clips", "sum"),
        )
        .reset_index()
    )
    acquisition_summary["_rank"] = acquisition_summary["Strategy"].map(STRATEGY_RANK)
    acquisition_summary = acquisition_summary.sort_values(["_rank", "Seed"]).drop(
        columns="_rank"
    )
    display(
        acquisition_summary.style.format(
            {
                "Mean_clips_per_batch": "{:.1f}",
                "Mean_events_per_1k_frames": "{:.2f}",
                "Mean_eventless_clips_pct": "{:.1f}",
                "Mean_unique_labels_per_batch": "{:.1f}",
            }
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for strategy in STRATEGY_ORDER:
        plot_data = acquisition_df[acquisition_df["Strategy"] == strategy]
        if plot_data.empty:
            continue
        color = STRATEGY_COLORS[strategy]
        axes[0].plot(
            plot_data["Query round"],
            plot_data["Events per 1k frames"],
            marker="o",
            color=color,
            label=strategy,
        )
        axes[1].plot(
            plot_data["Query round"],
            plot_data["Clips with no GT event (%)"],
            marker="o",
            color=color,
            label=strategy,
        )
    axes[0].set(
        title="Ground-truth event density of queried batches",
        xlabel="Query round",
        ylabel="Events per 1,000 frames",
    )
    axes[1].set(
        title="Ground-truth eventless clips in queried batches",
        xlabel="Query round",
        ylabel="Eventless clips (%)",
    )
    for ax in axes:
        ax.grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

### Cumulative label mix and source-match diversity

These diagnostics test whether a strategy is merely finding more events, over-concentrating on a few source matches, or skewing toward one high-level event type.

In [ ]:
selection_profile_rows = []
event_types = ["serve", "return", "stroke"]
for info in run_infos:
    selected_videos = seen_by_run.get(info["run"], set())
    if not selected_videos:
        continue
    annotations = [train_by_video[video] for video in selected_videos]
    source_counts = pd.Series(
        [row["video"].rsplit("_", 2)[0] for row in annotations]
    ).value_counts()
    source_shares = source_counts / source_counts.sum()
    effective_sources = 1.0 / float((source_shares**2).sum())
    type_counts = {event_type: 0 for event_type in event_types}
    total_events = 0
    for annotation in annotations:
        for event in annotation.get("events", []):
            total_events += 1
            tokens = set(event["label"].split("_"))
            for event_type in event_types:
                if event_type in tokens:
                    type_counts[event_type] += 1
                    break
    selection_profile_rows.append(
        {
            "Strategy": info["strategy"],
            "Seed": info["seed"],
            "Queried clips": len(annotations),
            "Effective source matches": effective_sources,
            "Top-5 source share (%)": 100 * source_shares.iloc[:5].sum(),
            "Serve events (%)": 100 * type_counts["serve"] / total_events,
            "Return events (%)": 100 * type_counts["return"] / total_events,
            "Stroke events (%)": 100 * type_counts["stroke"] / total_events,
        }
    )

selection_profile_df = pd.DataFrame(selection_profile_rows)
selection_profile_df["_rank"] = selection_profile_df["Strategy"].map(
    STRATEGY_RANK
)
selection_profile_df = selection_profile_df.sort_values(
    ["_rank", "Seed"]
).drop(columns="_rank")
display(
    selection_profile_df.style.format(
        {
            "Effective source matches": "{:.1f}",
            "Top-5 source share (%)": "{:.1f}",
            "Serve events (%)": "{:.1f}",
            "Return events (%)": "{:.1f}",
            "Stroke events (%)": "{:.1f}",
        }
    )
)

type_columns = [f"{event_type.title()} events (%)" for event_type in event_types]
plot_profile = selection_profile_df.set_index("Strategy").reindex(
    [strategy for strategy in STRATEGY_ORDER if strategy in set(selection_profile_df["Strategy"])]
)
ax = plot_profile[type_columns].plot(
    kind="bar",
    stacked=True,
    figsize=(9, 4.5),
    color=["#4c78a8", "#f28e2b", "#59a14f"],
)
ax.set(
    title="High-level event mix in all queried clips",
    xlabel="Acquisition strategy",
    ylabel="Share of queried ground-truth events (%)",
)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Event type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Dataset split audit

Exact duplicate or temporally overlapping clips would be direct leakage. Clips from the same source match in train and validation are a weaker but still important dependence: court, broadcast, players, and match-specific appearance can be shared. The official test split is match-disjoint, but earlier experiments in this repository already inspected it; a new grouped holdout or nested grouped cross-validation is needed for pristine confirmation.

In [ ]:
split_names = ["train", "val", "test"]
splits = {name: read_json(dataset_dir / f"{name}.json") for name in split_names}


def source_match(video):
    return video.rsplit("_", 2)[0]


def interval(video):
    match, start, end = video.rsplit("_", 2)
    return match, int(start), int(end)


split_inventory = pd.DataFrame(
    [
        {
            "Split": name,
            "Clips": len(rows),
            "Frames": sum(int(row["num_frames"]) for row in rows),
            "Events": sum(len(row.get("events", [])) for row in rows),
            "Source matches": len({source_match(row["video"]) for row in rows}),
        }
        for name, rows in splits.items()
    ]
)
display(split_inventory)

videos_by_split = {
    name: {row["video"] for row in rows} for name, rows in splits.items()
}
matches_by_split = {
    name: {source_match(row["video"]) for row in rows}
    for name, rows in splits.items()
}
intervals_by_split = {}
for name, rows in splits.items():
    grouped = {}
    for row in rows:
        match, start, end = interval(row["video"])
        grouped.setdefault(match, []).append((start, end))
    intervals_by_split[name] = grouped


def count_temporal_overlaps(left_name, right_name):
    count = 0
    left_groups = intervals_by_split[left_name]
    right_groups = intervals_by_split[right_name]
    for match in left_groups.keys() & right_groups.keys():
        for left_start, left_end in left_groups[match]:
            for right_start, right_end in right_groups[match]:
                if max(left_start, right_start) < min(left_end, right_end):
                    count += 1
    return count


split_overlap_rows = []
for left, right in combinations(split_names, 2):
    shared_matches = matches_by_split[left] & matches_by_split[right]
    split_overlap_rows.append(
        {
            "Split pair": f"{left} / {right}",
            "Exact duplicate clips": len(
                videos_by_split[left] & videos_by_split[right]
            ),
            "Temporally overlapping clips": count_temporal_overlaps(left, right),
            "Shared source matches": len(shared_matches),
            f"% {left} matches shared": (
                100 * len(shared_matches) / len(matches_by_split[left])
            ),
            f"% {right} matches shared": (
                100 * len(shared_matches) / len(matches_by_split[right])
            ),
        }
    )
display(
    pd.DataFrame(split_overlap_rows).style.format(
        {column: "{:.1f}" for column in pd.DataFrame(split_overlap_rows).columns if column.startswith("%")}
    )
)

## 8. Training and evaluator sanity checks

Two implementation issues affect every scientific comparison:

- The trainer uses ChainedScheduler for what its configuration describes as warm-up followed by cosine decay. Chaining advances both schedulers on every update; SequentialLR is the appropriate sequential construction.
- The event-F1 matcher does not charge a wrong composite-class prediction as a false positive when any differently labeled event is present at that frame. The minimal counterexample below receives one false negative but no false positive.

The LR trace replicates the configured scheduler without training the model. The metric counterexample mirrors the current event-counting conditions.

In [ ]:
import warnings

import torch
from torch.optim.lr_scheduler import (
    ChainedScheduler,
    CosineAnnealingLR,
    LinearLR,
    SequentialLR,
)

warnings.filterwarnings(
    "ignore",
    message="The epoch parameter.*was not necessary",
    category=UserWarning,
)

reference_config = configs_by_run[next(iter(configs_by_run))]["f3ed"]
dataset_samples = reference_config["epoch_num_frames"] // (
    reference_config["clip_len"] * reference_config["stride"]
)
steps_per_epoch = math.ceil(dataset_samples / reference_config["batch_size"])
warmup_steps = reference_config["warm_up_epochs"] * steps_per_epoch
total_steps = reference_config["num_epochs"] * steps_per_epoch
cosine_steps = total_steps - warmup_steps


def trace_lr(schedule_kind):
    parameter = torch.nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.SGD(
        [parameter], lr=reference_config["learning_rate"]
    )
    warmup = LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=warmup_steps,
    )
    cosine = CosineAnnealingLR(optimizer, cosine_steps)
    if schedule_kind == "current ChainedScheduler":
        scheduler = ChainedScheduler([warmup, cosine])
    else:
        scheduler = SequentialLR(
            optimizer,
            schedulers=[warmup, cosine],
            milestones=[warmup_steps],
        )
    trace = [optimizer.param_groups[0]["lr"]]
    for _ in range(total_steps):
        optimizer.step()
        scheduler.step()
        trace.append(optimizer.param_groups[0]["lr"])
    return np.asarray(trace)


current_lr = trace_lr("current ChainedScheduler")
sequential_lr = trace_lr("intended SequentialLR")
epoch_axis = np.arange(total_steps + 1) / steps_per_epoch
sample_epochs = [0, reference_config["warm_up_epochs"], 47, 50]
lr_check = pd.DataFrame(
    {
        "Epoch": sample_epochs,
        "Current chained LR": [
            current_lr[min(epoch * steps_per_epoch, total_steps)]
            for epoch in sample_epochs
        ],
        "Sequential warmup/cosine LR": [
            sequential_lr[min(epoch * steps_per_epoch, total_steps)]
            for epoch in sample_epochs
        ],
    }
)
display(lr_check.style.format({"Current chained LR": "{:.8f}", "Sequential warmup/cosine LR": "{:.8f}"}))

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(epoch_axis, current_lr, label="Current ChainedScheduler", linewidth=2)
ax.plot(epoch_axis, sequential_lr, label="Sequential warm-up then cosine", linewidth=2)
ax.axvline(reference_config["warm_up_epochs"], color="black", linestyle="--", alpha=0.5)
ax.set(title="Configured learning-rate schedules", xlabel="Epoch", ylabel="Learning rate")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


def current_event_counts(labels, predictions, delta=0):
    counts = {}
    for index, prediction in enumerate(predictions):
        local_labels = labels[
            max(0, index - delta):min(len(labels), index + delta + 1)
        ]
        local_predictions = predictions[
            max(0, index - delta):min(len(predictions), index + delta + 1)
        ]
        if prediction > 0 and prediction in local_labels:
            counts.setdefault(prediction, [0, 0, 0])[0] += 1
        if prediction > 0 and sum(local_labels) == 0:
            counts.setdefault(prediction, [0, 0, 0])[1] += 1
        if labels[index] > 0 and labels[index] not in local_predictions:
            counts.setdefault(labels[index], [0, 0, 0])[2] += 1
    return counts


example_labels = [0, 1, 0]
example_predictions = [0, 2, 0]
bug_counts = current_event_counts(example_labels, example_predictions)
metric_counterexample = pd.DataFrame(
    [
        {"Class": "true class 1", "Current TP": bug_counts.get(1, [0, 0, 0])[0], "Current FP": bug_counts.get(1, [0, 0, 0])[1], "Current FN": bug_counts.get(1, [0, 0, 0])[2], "Expected FP": 0, "Expected FN": 1},
        {"Class": "wrong predicted class 2", "Current TP": bug_counts.get(2, [0, 0, 0])[0], "Current FP": bug_counts.get(2, [0, 0, 0])[1], "Current FN": bug_counts.get(2, [0, 0, 0])[2], "Expected FP": 1, "Expected FN": 0},
    ]
)
display(Markdown("**Wrong-class-at-true-event counterexample**"))
display(metric_counterexample)
assert bug_counts == {1: [0, 0, 1]}


## 9. Methodology audit

Prioritized findings:

1. **Critical — the evaluator and LR schedule need correction before more strategy runs.** The counterexample above shows that event F1 omits the false positive for a wrong-class prediction at a true-event frame. The evaluator also ignores its window argument and hardcodes NMS window 5. ChainedScheduler advances warm-up and cosine concurrently, reaches the cosine minimum around epoch 47, and then rises again. These issues make the current F1 values invalid and the training schedule different from its intended design; fix them, add unit tests, and rerun all compared methods.

2. **Critical — strategy is confounded with training noise.** The optimized trainer enables cuDNN benchmarking and disables deterministic algorithms in [train_f3ed_f3set_tennis_optimized.py](../../scripts/train_f3ed_f3set_tennis_optimized.py). Identical round-0 annotations nevertheless produce a performance spread comparable to or larger than the observed strategy effects. A separate trained model for each strategy therefore does not isolate acquisition quality.

3. **Critical — one seed cannot answer whether a strategy works.** The shell driver currently runs seed 0 only. Curves, rounds, queried clips, and validation epochs are not independent replicates.

4. **High — validation is dependent and repeatedly reused.** There are no duplicate or overlapping train/validation clips, which is good, but nearly all validation clips come from source matches also present in training. Every round searches epochs 30–49 by validation edit, and these same histories are then used to choose metrics and strategies. The existing test split is match-disjoint, but it was already inspected in the initial-pool experiment, so it is development data rather than a pristine final holdout. Use a new match-grouped holdout or nested grouped cross-validation for confirmation.

5. **High — fine uncertainty is gated by coarse detections and structurally misaligned.** In [calculate_query_scores](../../scripts/train_f3ed_f3set_tennis.py), fine scores use only the predicted-event mask. Missed events are invisible; logits at false-positive event locations were not supervised as fine labels; and 29 Bernoulli uncertainties are averaged despite mutually exclusive and conditional decoder groups. This can assign zero to an informative clip or favor ambiguous false positives.

6. **High — checkpoint selection is mismatched to the reported F1 readout.** Checkpoints maximize validation edit; event and element F1 are simply the values at that epoch. If F1 is the actual target, pre-register a matching checkpoint criterion or report edit as primary and F1 as secondary.

7. **Medium — active and random branches consume different RNG streams.** Active runs perform an extra scoring DataLoader pass; random runs do not. Reset model/training seeds per round and give training, acquisition, and DataLoaders independent generators.

8. **Medium — UM and EM are largely redundant and not calibrated epistemic uncertainty.** The score table quantifies their near-identical rankings. Coarse probabilities come from class-weighted cross-entropy, so 0.5 is a cost-sensitive boundary rather than a calibrated posterior boundary. Do not spend on a large UM/EM grid unless same-checkpoint batches differ materially.

9. **Medium — the acquisition universe and cost model are optimistic.** Every training query unit already contains at least one annotated event and uses oracle point boundaries. The unlabeled JSON also retains event labels; current inference does not read them, but stripping them is a safer invariant. Handedness metadata is used during inference and should be declared available before annotation. Whole-clip frame budgets are internally consistent, yet event-label complexity, clip setup cost, and review time may better approximate human effort.

10. **Design choice to document — retraining and fixed update count.** Each round trains from scratch for a fixed number of sampled frames. This controls optimizer updates but changes the effective number of passes over each labeled pool. A convergence or warm-start ablation would show whether the conclusion depends on that choice.

Positive checks: this experiment does not evaluate test during training; all current strategies start from the exact same annotation pool; realized frame budgets are close; round transitions and saved top-score selections are internally consistent.

## 10. Recommended next experiments

Run these in order.

| Priority | Experiment | Minimal design | Decision it unlocks |
|---|---|---|---|
| P0 | Correctness pass | Fix and unit-test event F1, honor the NMS window argument, replace ChainedScheduler with SequentialLR, log the LR trace, strip labels from unlabeled artifacts, and record code/data/environment hashes. Do not compare new strategy runs until this passes. | Make the training/evaluation target trustworthy. |
| P0 | Reproducibility harness | With the corrected code, train the identical saved round-0 pool at least 3 times. First use deterministic settings; then repeat optimized settings only if its speed benefit warrants quantified variance. Reset documented seeds for initialization, augmentation, workers, and every round. | Establish whether acquisition effects can be resolved above optimizer/GPU noise. |
| P0 | Fair acquisition fork | For each seed, train round 0 once, save the checkpoint and RNG-independent pool state, then fork that exact checkpoint to every query rule. On later rounds use common per-round training seeds across strategies. | Isolate the query rule from the model that happened to generate its scores. |
| P1 | Paired replicated comparison | At least 5 paired seeds over 5% to 10%: Random, one coarse representative (provisionally Coarse UM), a repaired fine rule, and a coarse/fine hybrid. Pre-register edit AULC or the actual target metric. Report per-seed curves and paired differences. | Decide whether any active rule reliably beats random. |
| P1 | Repair fine acquisition | Compare current hard gating with soft coarse-probability weighting over all frames, cost-normalized summed utility, decoder-aligned grouped categorical entropy, and a coarse/fine hybrid. Use stored snapshots for a cheap counterfactual ranking audit before retraining. | Test whether hard gating and unstructured averaging explain Fine UM's low, serve-heavy event yield. |
| P1 | Pooling ablation | Compare MEAN with MAX or top-k mean on the same model snapshots, then train only materially different batch selections. | Check whether mean pooling washes out sparse uncertain event frames. |
| P2 | Baselines and budget | Add a fixed 5% no-query lower bound, multiple random trajectories, a predicted-event-density baseline, an oracle event-density diagnostic ceiling, and a full-data upper bound. Extend beyond 10% only after a replicated rule beats random in the current window. | Separate uncertainty from simply acquiring more event labels and put saturation in context. |
| P2 | Diversity-aware batch | Compare uncertainty-only ranking with uncertainty plus embedding diversity or match-level caps. Measure duplicate labels/source matches in each batch. | Reduce redundancy within the roughly 1%-of-frames batch. |
| P3 | Frozen confirmation | Freeze strategy, pooling, checkpoint metric, budgets, and seeds; evaluate on a newly reserved match-grouped holdout or use nested grouped cross-validation. Report the existing official test only as an already-seen development benchmark. | Obtain a defensible unseen-match generalization result. |

A compute-conscious next grid is **5 seeds × 4 strategies = 20 trajectories**, with identical per-seed initial pools and controlled per-round training randomness. Complete the already-running Fine EM trajectory for bookkeeping, but do not launch replicated UM/EM grids until the score-overlap analysis justifies keeping both.

## 11. Bottom line

In [ ]:
bottom_lines = [
    "**Current decision:** no query strategy is yet proven to work better than random.",
    "",
]
for strategy in ["Coarse UM", "Coarse EM", "Fine UM", "Fine EM"]:
    strategy_data = post_query_df[post_query_df["strategy"] == strategy]
    if strategy_data.empty:
        bottom_lines.append(f"- **{strategy}:** insufficient post-query results.")
        continue
    latest = (
        strategy_data.sort_values("round").groupby("seed", as_index=False).tail(1)
    )
    edit_delta = latest["delta_val_edit"].mean()
    event_delta = 100 * latest["delta_val_f1_event"].mean()
    element_delta = 100 * latest["delta_val_f1_element"].mean()
    if strategy.startswith("Coarse") and event_delta > 0 and element_delta > 0:
        assessment = "promising F1 signal, unconfirmed"
    elif event_delta < 0 and element_delta < 0:
        assessment = "not working in its current form"
    else:
        assessment = "mixed result"
    bottom_lines.append(
        f"- **{strategy}: {assessment}.** Latest deltas versus random: "
        f"{edit_delta:+.2f} edit, {event_delta:+.2f} event-F1 points, "
        f"{element_delta:+.2f} element-F1 points."
    )

if not score_diag_df.empty:
    bottom_lines.extend(
        [
            "",
            (
                "The acquisition implementation passes the saved ranking and pool-transition "
                "checks. The main blockers are evaluator/scheduler correctness, experimental "
                "control, and statistical power, "
                "not an obvious top-k selection bug."
            ),
        ]
    )
if not acquisition_df.empty and {
    "Random",
    "Fine UM",
}.issubset(set(acquisition_summary["Strategy"])):
    comp = acquisition_summary.set_index("Strategy")
    fine_yield = comp.loc["Fine UM", "Mean_events_per_1k_frames"]
    random_yield = comp.loc["Random", "Mean_events_per_1k_frames"]
    bottom_lines.append(
        f"Fine UM selected **{fine_yield:.1f}** ground-truth events per 1,000 frames "
        f"versus **{random_yield:.1f}** for random, a concrete lead for the fine-gating ablation."
    )

bottom_lines.extend(
    [
        "",
        (
            "The immediate next move is to fix and unit-test the evaluator and LR schedule, "
            "then control round-level randomness and run paired replicates—not to extend "
            "the annotation budget or multiply nearly redundant UM/EM variants."
        ),
    ]
)
display(Markdown("\n".join(bottom_lines)))